In [ ]:
import torch

from occhio import ToyModel
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions import CorrelatedPairs
from occhio.model_grid import ModelGrid, Axis, TrainingAxis
from occhio.visualization import (
    plot_embedding,
)

In [ ]:
device = "mps"

In [ ]:
N_FEATURES = 10
N_HIDDEN = 2

In [ ]:
def create_model(params):
    generator = torch.Generator(device=device).manual_seed(199)

    return ToyModel(
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device, generator=generator),
        distribution=CorrelatedPairs(
            N_FEATURES,
            density=1 - params["Sparsity"],
            correlation=params["Correlation"],
            device=device,
            generator=generator,
        ),
        # Correlated pairs of diminishing importance, where within the second feature is also less important.
        importances=torch.tensor([1, 0.7] * int(N_FEATURES / 2))
        * (0.9 ** torch.arange(N_FEATURES)),
        device=device,
    )


grid = ModelGrid(
    create_model,
    axes=[
        Axis(label="Sparsity", values=[0.0, 0.8, 0.9, 0.99]),
        Axis(label="Correlation", values=[0.0, 0.5, 1]),
        # Axis(label="Importance", values=[1, 0.9, 0.7]),
    ],
)

In [ ]:
grid.fit(4000)

In [ ]:
plot_embedding(grid)

In [ ]:
import importlib
import occhio.visualization_2.core
import occhio.visualization_2.plots

importlib.reload(occhio.visualization_2.core)
importlib.reload(occhio.visualization_2.plots)
from occhio.visualization_2.plots.embedding import EmbeddingPlot

plot_embedding_2 = EmbeddingPlot()
plot_embedding_2(grid)

In [ ]:
import importlib
import occhio.visualization_2.core.base
import occhio.visualization_2.embedding
import occhio.visualization_2.representation

importlib.reload(occhio.visualization_2.base)
importlib.reload(occhio.visualization_2.embedding)
importlib.reload(occhio.visualization_2.representation)
from occhio.visualization_2.representation import RepresentationPlot

plot_representation_2 = RepresentationPlot()
plot_representation_2(grid)

In [ ]:
import importlib

import occhio.visualization_2.core.base_plot
import occhio.visualization_2.core.composite_plot
import occhio.visualization_2.plots.representation
import occhio.visualization_2.plots.embedding

importlib.reload(occhio.visualization_2.core.base_plot)
importlib.reload(occhio.visualization_2.core.composite_plot)
importlib.reload(occhio.visualization_2.plots.representation)
importlib.reload(occhio.visualization_2.plots.embedding)

# Import AFTER reload
from occhio.visualization_2.core.composite_plot import CompositePlot, Span
from occhio.visualization_2.plots import EmbeddingPlot, RepresentationPlot

composite = CompositePlot(
    layout=[
        [Span(EmbeddingPlot(), rowspan=2), RepresentationPlot()],
        [None, RepresentationPlot()],
    ],
    # column_widths=[1],
    # row_heights=[2, 1],
)

# composite = CompositePlot(
#     layout=[
#         (RepresentationPlot(), 1, 1),
#         (EmbeddingPlot(), 2, 1),
#     ],
#     column_widths=[1],
# )

In [ ]:
composite(grid)